# Graph: inductive vs transductive — mechanism analysis

Why does the bipartite graph behave so differently relative to the no-graph XGBoost baseline across the two `full_full` regimes? This notebook isolates the mechanism with the **raw-ESM concat ablation** (`--concat_raw_prot`).

**Setup recap.** The graph probe is `[ChemBERTa ‖ z_prot]`, where `z_prot` is the graph-enriched protein embedding — a trained **256-d proj+ReLU bottleneck** of ESM. Even `history` only concatenates *stages of that bottleneck*, never the raw 1280-d ESM. The baseline, by contrast, feeds XGBoost the full raw ESM. `rawp` puts the raw ESM back next to `z_prot`: `[ChemBERTa ‖ raw ESM-1280 ‖ z_prot]`.

**Reading the result.**
- *transductive*: raw ESM lifts the graph (+0.01–0.03 AUROC) → most of the graph's deficit vs boosting is just the **bottleneck hiding raw ESM**. But `rawp` still doesn't beat baseline → `z_prot` adds nothing over raw ESM here (graph ≈ lossy compression; extra columns even dilute slightly).
- *inductive*: pure graph beats `rawp` and baseline → on cold molecules the graph's binding-profile signal **genuinely transfers**, and raw-ESM identity memorization is a liability.

Data: `train_graph_full_full.py --concat_raw_prot`. See `README.md` in this folder.

In [110]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
from orbind.dataset import metrics

FOLD = 1   # which LORAX random fold to display (1..5)
CKPT_DIR = ROOT / "results" / "full_full" / "checkpoints"
BASELINE_CSV = ROOT / "results" / "full_full" / "tables" / "baselines.csv"
REGIMES = ["transductive", "inductive_molecule"]
PLOT_METRICS = ["AUROC", "AUPRC", "MCC", "F1"]
print("ckpt_dir =", CKPT_DIR, "| fold =", FOLD)

ckpt_dir = ..\results\full_full\checkpoints | fold = 1


In [111]:
# Load every full_full checkpoint for FOLD, keyed by (regime, arch, variant).
# variant = mp_mode [+ _q##] [+ _history]  — fully identifies a run.
def variant_of(ckpt):
    c = ckpt["config"]; v = c["mp_mode"]
    q = c.get("mol_quality_q", 0) or 0
    if q > 0: v += f"_q{int(q*100)}"
    if c.get("history", False):
        v += "_history" if c.get("history_depth", 2) >= 2 else "_hist1"
    if c.get("concat_raw_prot", False): v += "_rawp"
    return v

records, preds = [], {}
for path in sorted(CKPT_DIR.glob(f"*_fold{FOLD}.pt")):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    regime, arch, var = ckpt["regime"], ckpt["arch"], variant_of(ckpt)
    _, test_y = ckpt["test_sup"]
    y, scores = test_y.numpy(), ckpt["test_scores"]
    records.append({"regime": regime, "arch": arch, "variant": var, **metrics(y, scores)})
    preds[(regime, arch, var)] = (y, scores)

df = (pd.DataFrame(records).set_index(["regime", "arch", "variant"]).sort_index().round(3)
      if records else pd.DataFrame())
print(f"{len(preds)} checkpoints loaded")
display(df)

36 checkpoints loaded


AUROC  AUPRC    MCC     F1  \
regime             arch variant                                          
inductive_molecule gat  all_edges           0.740  0.590  0.456  0.534   
                        pos_only            0.730  0.612  0.475  0.533   
                        signed              0.746  0.607  0.496  0.562   
                        signed_history      0.772  0.636  0.483  0.552   
                        signed_q87          0.793  0.629  0.483  0.557   
                        signed_q95          0.804  0.648  0.504  0.595   
                        signed_q95_hist1    0.783  0.645  0.500  0.566   
                        signed_q95_history  0.794  0.649  0.501  0.562   
                        signed_q99          0.756  0.545  0.274  0.455   
                   gnn  all_edges           0.767  0.613  0.468  0.551   
                        pos_only            0.737  0.611  0.495  0.557   
                        signed              0.780  0.643  0.492  0.563   
                        signed_history      0.782  0.640  0.510  0.577   
                        signed_q87          0.817  0.627  0.477  0.552   
                        signed_q87_hist1    0.785  0.639  0.496  0.567   
                        signed_q87_history  0.796  0.639  0.508  0.577   
                        signed_q95          0.783  0.641  0.486  0.561   
                        signed_q99          0.792  0.635  0.496  0.574   
transductive       gat  all_edges           0.794  0.642  0.534  0.642   
                        pos_only            0.790  0.696  0.555  0.640   
                        signed              0.799  0.640  0.573  0.660   
                        signed_history      0.801  0.657  0.598  0.677   
                        signed_q87          0.814  0.622  0.563  0.659   
                        signed_q95          0.832  0.639  0.540  0.648   
                        signed_q95_hist1    0.864  0.716  0.584  0.679   
                        signed_q95_history  0.863  0.710  0.597  0.689   
                        signed_q99          0.764  0.560  0.315  0.492   
                   gnn  all_edges           0.838  0.699  0.564  0.666   
                        pos_only            0.784  0.681  0.592  0.670   
                        signed              0.810  0.638  0.574  0.661   
                        signed_history      0.793  0.639  0.595  0.677   
                        signed_q87          0.821  0.660  0.572  0.661   
                        signed_q87_hist1    0.840  0.682  0.601  0.687   
                        signed_q87_history  0.834  0.669  0.598  0.682   
                        signed_q95          0.853  0.694  0.593  0.685   
                        signed_q99          0.852  0.709  0.569  0.670   

                                            precision  recall  
regime             arch variant                                
inductive_molecule gat  all_edges               0.695   0.434  
                        pos_only                0.752   0.413  
                        signed                  0.747   0.450  
                        signed_history          0.733   0.442  
                        signed_q87              0.719   0.455  
                        signed_q95              0.681   0.529  
                        signed_q95_hist1        0.748   0.455  
                        signed_q95_history      0.761   0.446  
                        signed_q99              0.399   0.529  
                   gnn  all_edges               0.689   0.459  
                        pos_only                0.754   0.442  
                        signed                  0.730   0.459  
                        signed_history          0.753   0.467  
                        signed_q87              0.712   0.450  
                        signed_q87_hist1        0.732   0.463  
                        signed_q87_history      0.745   0.471  
                        signed_q95              0.721   0.459  
               

In [112]:
# No-graph XGBoost baseline on the SAME splits (both regimes) — the bar to beat
bl = pd.read_csv(BASELINE_CSV).set_index("regime")
BASE = {r: {m: float(bl.loc[r, m]) for m in PLOT_METRICS} for r in REGIMES}
for r in REGIMES:
    print(f"{r:20s} no-graph boost:", {k: round(v, 3) for k, v in BASE[r].items()})

transductive         no-graph boost: {'AUROC': 0.891, 'AUPRC': 0.755, 'MCC': 0.644, 'F1': 0.727}
inductive_molecule   no-graph boost: {'AUROC': 0.77, 'AUPRC': 0.621, 'MCC': 0.473, 'F1': 0.557}


In [113]:
# Reusable: rows = regimes, cols = metrics; bars = listed (arch, variant) + baseline
def plot_grid(specs, suptitle, savename, rot=0, fs=7):
    """specs: list of (display_name, color, arch, variant)."""
    names  = [s[0] for s in specs] + ["XGBoost\n(no graph)"]
    colors = [s[1] for s in specs] + ["#A0A0A0"]
    fig, axes = plt.subplots(len(REGIMES), len(PLOT_METRICS),
                              figsize=(1.0*len(names)*len(PLOT_METRICS), 3.8*len(REGIMES)))
    axes = np.array(axes).reshape(len(REGIMES), len(PLOT_METRICS))
    for ri, regime in enumerate(REGIMES):
        for mi, metric in enumerate(PLOT_METRICS):
            ax = axes[ri, mi]; base = BASE[regime][metric]
            vals = [df.loc[(regime, a, v), metric] if (regime, a, v) in df.index else float("nan")
                    for _, _, a, v in specs] + [base]
            bars = ax.bar(names, vals, color=colors)
            ax.axhline(base, color="#A0A0A0", ls="--", lw=1)
            ax.set_title(f"{regime} — {metric}", fontsize=9)
            ax.set_ylim(0, 1); ax.tick_params(axis="x", labelsize=fs, rotation=rot)
            for b, v in zip(bars, vals):
                if not np.isnan(v):
                    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.3f}",
                            ha="center", va="bottom", fontsize=fs)
    fig.suptitle(suptitle, fontsize=12, y=1.01); plt.tight_layout()
    (ROOT/"notebooks/figures").mkdir(parents=True, exist_ok=True)
    plt.savefig(ROOT/f"notebooks/figures/{savename}.png", dpi=150, bbox_inches="tight")
    plt.show()

GNN_C = {"pos_only":"#4878CF","all_edges":"#6ACC65","signed":"#D65F5F"}
GAT_C = {"pos_only":"#3A5FA0","all_edges":"#4E9E4E","signed":"#9467BD"}

In [ ]:
# G: does the graph ADD signal, or just COMPRESS away raw ESM?
# Probe on [ChemBERTa || RAW ESM-1280 || z_prot] (rawp) vs graph-only vs baseline.
# Prediction: if graph only compresses -> rawp == baseline in both regimes.
#             if graph adds transferable signal -> rawp >= baseline, esp. inductive.
specs = [("gnn signed",          "#D65F5F", "gnn", "signed"),
         ("gnn signed +rawp",    "#7F1518", "gnn", "signed_rawp"),
         ("gnn q95 +rawp",       "#B03A1E", "gnn", "signed_q95_rawp"),
         ("gnn q87 +rawp",       "#E06C2A", "gnn", "signed_q87_rawp"),
         ("gat signed +rawp",    "#5E3D86", "gat", "signed_rawp"),
         ("gat q95 h1 +rawp",    "#9467BD", "gat", "signed_q95_hist1_rawp")]
plot_grid(specs, f"G: raw-ESM concat ablation — graph vs +rawp vs no-graph (fold {FOLD})",
          "graph_full_full_rawp", fs=6.5)

## Вывод

- **transductive-проигрыш графа бустингу — артефакт бутылочного горла**, а не «бустинг умнее»: вернёшь сырой ESM (`rawp`) — граф подрастает на 2/3 разрыва. Остаток — разбавление сильного сырого сигнала лишними `z_prot`-колонками.
- **inductive-выигрыш графа — реальный переносимый сигнал**: сырой ESM тут даже вредит (identity-память не обобщается на холодные молекулы).
- Т.е. «модель vs бустинг» — оба фактора, и `rawp` их чисто разделяет: transductive = недообученность/компрессия, baseline-edge = leak на identity.